In [1]:

import pandas as pd
import numpy as np


In [2]:
from datetime import date, timedelta

In [ ]:
dfpreclean = pd.read_csv('Transactions/Paypal_Transactions3.csv')
dfpreclean.head(5)

,Transaction_ID,Type,Transaction_Type,Customer_Name,Total,Success,Day,Transaction_Notes,Source,Country,Auth_code
0,1234567,Charge,Goods and Services,James,3286,1,1/2/2023,Thanks,Tablet,US,X8JZG7YH
1,9876543,Refund,Friends & Family,Emily,1624,1,1/3/2023,Raffle,Phone,US,D2F3R6KP
2,4567890,Charge,Goods and Services,Liam,2659,1,1/4/2023,Thanks,Desktop,US,Q9L4T1VW
3,7654321,Charge,Goods and Services,Olivia,4897,1,1/5/2023,Thanks,Phone,US,M7N5P0QI
4,2345678,Charge,Friends & Family,Benjamin,3643,1,1/6/2023,Thanks,Desktop,UK,B6K8D3XJ


In [4]:
dfpreclean.drop(['Transaction_ID', 'Auth_code'],axis=1, inplace= True)

In [5]:
dfpreclean2 = dfpreclean[dfpreclean['Success']==1]

In [6]:
dfpreclean2['Transaction_Notes'].fillna("N/A", inplace=True)

C:\Users\😘\AppData\Local\Temp\ipykernel_1784\4233614566.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfpreclean2['Transaction_Notes'].fillna("N/A", inplace=True)
C:\Users\😘\AppData\Local\Temp\ipykernel_1784\4233614566.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfpreclean2['Transaction_Notes'].fillna("N/A", inplace=True)


In [7]:
dfpreclean2['Day'] = pd.to_datetime(dfpreclean2['Day'])

C:\Users\😘\AppData\Local\Temp\ipykernel_1784\3694656972.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfpreclean2['Day'] = pd.to_datetime(dfpreclean2['Day'])


In [8]:
dfpreclean2.columns

Index(['Type', 'Transaction_Type', 'Customer_Name', 'Total', 'Success', 'Day',
       'Transaction_Notes', 'Source', 'Country'],
      dtype='object')

#### create a new dataframe 'df' and drop the column 'success' by simplying not including it

In [9]:
df = dfpreclean2.loc[:, ['Total', 'Transaction_Type', 'Type', 'Country','Source', 'Day', 'Customer_Name', 'Transaction_Notes']]

In [10]:
df.head()

,Total,Transaction_Type,Type,Country,Source,Day,Customer_Name,Transaction_Notes
0,3286,Goods and Services,Charge,US,Tablet,2023-01-02,James,Thanks
1,1624,Friends & Family,Refund,US,Phone,2023-01-03,Emily,Raffle
2,2659,Goods and Services,Charge,US,Desktop,2023-01-04,Liam,Thanks
3,4897,Goods and Services,Charge,US,Phone,2023-01-05,Olivia,Thanks
4,3643,Friends & Family,Charge,UK,Desktop,2023-01-06,Benjamin,Thanks


### We will now proceed to get all the calculations that we will need

In [11]:
totalsum = np.sum(df['Total'])

In [12]:
total_transactions = df['Type'].count()
total_transactions

np.int64(195)

In [13]:
mean_transaction = np.mean(df['Total'])
median_transaction = np.median(df['Total'])
max_transaction = np.max(df['Total'])

In [14]:
total_unique_customers = df['Customer_Name'].nunique()
total_unique_customers

37

In [15]:
chargeonlytransactions = df[df['Type'] =='Charge']

In [16]:
refundonlytransactions = df[df['Type'] =='Refund']
chargebackonlytransactions = df[df['Type'] =='Chargeback']

In [17]:
days90 = pd.to_datetime(date.today() - timedelta(days=90))
days180 = pd.to_datetime(date.today() - timedelta(days=180))

In [18]:
chargetotal =np.sum(chargeonlytransactions['Total'])

In [19]:
charge90days = np.sum(chargeonlytransactions[chargeonlytransactions['Day']> days90]['Total'])

In [20]:
charge180days = np.sum(chargeonlytransactions[chargeonlytransactions['Day']> days180]['Total'])

In [21]:
refundtotal =np.sum(refundonlytransactions['Total'])
refund90days = np.sum(refundonlytransactions[refundonlytransactions['Day']> days90]['Total'])
refund180days = np.sum(refundonlytransactions[refundonlytransactions['Day']> days180]['Total'])

In [22]:
chargebacktotal =np.sum(chargebackonlytransactions['Total'])
chargeback90days = np.sum(chargebackonlytransactions[chargebackonlytransactions['Day']> days90]['Total'])
chargeback180days = np.sum(chargebackonlytransactions[chargebackonlytransactions['Day']> days180]['Total'])

In [23]:
refundratelifetime = (refundtotal/chargetotal)
refundrate90days = (refund90days/charge90days) if charge90days != 0 else 0
refundrate180days = (refund180days/charge180days) if charge180days != 0 else 0

In [24]:
chargebackratelifetime = (chargebacktotal/chargetotal)
chargebackrate90days = (chargeback90days/charge90days) if charge90days != 0 else 0
chargebackrate180days = (chargeback180days/charge180days) if charge180days != 0 else 0

In [25]:
pivottablenames = pd.pivot_table(df, index=['Customer_Name'], aggfunc={'Total':np.sum, 'Customer_Name': 'count',})
pivottablenames = pivottablenames.rename(columns={'Customer_Name': 'count_of_total', 'Total': 'sum_of_total'})
pivottablenames= pivottablenames.loc[:, ['sum_of_total', 'count_of_total']]

C:\Users\😘\AppData\Local\Temp\ipykernel_1784\477021026.py:1: FutureWarning: The provided callable <function sum at 0x00000134B585CA40> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  pivottablenames = pd.pivot_table(df, index=['Customer_Name'], aggfunc={'Total':np.sum, 'Customer_Name': 'count',})


In [26]:
avg_transactions_count_per_customer = np.mean(pivottablenames['count_of_total'])

In [27]:
avg_transactions_sum_per_customer = np.mean(pivottablenames['sum_of_total'])

In [28]:

pivottabltransactiontype = pd.pivot_table(df, index=['Transaction_Type'], aggfunc={'Transaction_Type': 'count', 'Total': np.sum})
pivottabltransactiontype['totalpercent'] = (pivottabltransactiontype['Total']/totalsum).apply('{:.2%}'.format)

C:\Users\😘\AppData\Local\Temp\ipykernel_1784\3065341047.py:1: FutureWarning: The provided callable <function sum at 0x00000134B585CA40> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  pivottabltransactiontype = pd.pivot_table(df, index=['Transaction_Type'], aggfunc={'Transaction_Type': 'count', 'Total': np.sum})


In [29]:
pivottabltransactioncountry = pd.pivot_table(df, index=['Country'], aggfunc={'Country': 'count', 'Total': np.sum})
pivottabltransactioncountry['totalpercent'] = (pivottabltransactioncountry['Total']/totalsum).apply('{:.2%}'.format)

C:\Users\😘\AppData\Local\Temp\ipykernel_1784\3291236946.py:1: FutureWarning: The provided callable <function sum at 0x00000134B585CA40> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  pivottabltransactioncountry = pd.pivot_table(df, index=['Country'], aggfunc={'Country': 'count', 'Total': np.sum})


In [30]:
firstname = 'Ryan'
namefinal = df[df['Customer_Name'].str.contains(firstname, case= False)]

In [31]:
payment_note = df[df['Transaction_Notes'].isna() == False]
flagged_words = 'raffle|razz|lottery'
payment_note_final = df[df['Transaction_Notes'].str.contains(flagged_words, case=False)]

In [32]:
highticketval =3500
highticket = df[df['Total'] >= highticketval].copy()
highticket = highticket.sort_values(by='Total', ascending=False)

In [33]:
dup =df.copy()

In [34]:
dup['Customer_Name_next'] = dup['Customer_Name'].shift(1)
dup['Customer_Name_prev'] = dup['Customer_Name'].shift(-1)


In [35]:
dup['created_at_day'] = dup['Day']
dup['created_at_dayprev'] = dup['Day'].shift(-1)
dup['created_at_daynext'] = dup['Day'].shift(1)

In [36]:
dup3 = dup.query('(created_at_day == created_at_dayprev | created_at_day == created_at_daynext) & (Customer_Name == Customer_Name_next | Customer_Name == Customer_Name_prev)')


In [37]:
dfcalc = pd.DataFrame({'totalsum':[totalsum],
                           'mean_transaction':[mean_transaction],
                           'median_transaction':[median_transaction], 
                           'max_transaction':[max_transaction],
                           'total_transactions':[total_transactions],
                           'chargetotal':[chargetotal],
                           'charge90days':[charge90days],
                           'charge180days':[charge180days],
                           'refundtotal':[refundtotal],
                           'refund90days':[refund90days],
                           'refund180days':[refund180days],
                           'refundratelifetime':[refundratelifetime],
                           'refundrate90days':[refundrate90days],
                           'refundrate180days':[refundrate180days],
                           'chargebacktotal':[chargebacktotal],
                           'chargeback90days':[chargeback90days],
                           'chargeback180days':[chargeback180days],
                           'chargebackratelifetime':[chargebackratelifetime],
                           'chargebackrate90days':[chargebackrate90days],
                           'chargebackrate180days':[chargebackrate180days],
                           'total_unique_customer_names':[total_unique_customers],                      
                           'avg_transactions_count_per_customer_name':[avg_transactions_count_per_customer],
                           'avg_transactions_sum_per_customer_name':[avg_transactions_sum_per_customer],
                           '90 Days':[days90],
                           '180 Days':[days180],
                        })


In [38]:
format_mapping = {"totalsum": '${:,.2f}',
                  "mean_transaction": '${:,.2f}',
                  "median_transaction": '${:,.2f}',
                  "max_transaction": '${:,.2f}',
                  "total_transactions": '{:,.0f}', 
                  'chargetotal': '${:,.2f}',
                  'charge90days': '${:,.2f}',
                  'charge180days': '${:,.2f}',
                  'refundtotal': '${:,.2f}',
                  'refund90days': '${:,.2f}',
                  'refund180days': '${:,.2f}',
                  'refundratelifetime':'{:.2%}',
                  'refundrate90days':'{:.2%}',
                  'refundrate180days':'{:.2%}',
                  'chargebacktotal':'${:,.2f}',
                  'chargeback90days':'${:,.2f}',
                  'chargeback180days':'${:,.2f}',
                  'chargebackratelifetime':'{:.2%}',
                  'chargebackrate90days':'{:.2%}',
                  'chargebackrate180days':'{:.2%}',
                  "total_unique_customer_names": '{:,.0f}',
                  "avg_transactions_count_per_customer_name": '{:,.2f}',
                  "avg_transactions_sum_per_customer_name": '${:,.2f}',                  
                    }

In [39]:

for key, value in format_mapping.items():
            dfcalc[key] = dfcalc[key].apply(value.format)